In [1]:
%load_ext autoreload
%autoreload 2

In [26]:
import pandas as pd
from preprocess import fromat_vendor_name, format_shop_category_name, process_description, process_vendor_name, process_shop_category_name, extract_description
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


In [3]:
df = pd.read_csv('train.tsv', sep='\t')

In [4]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(['category_id', 'department_id'], axis=1),
                                                    df[['category_id', 'department_id']],
                                                    test_size=0.2, random_state=42)

In [8]:
top_vendor_names = fromat_vendor_name(X_train)["vendor_name"].value_counts().head(11).index.to_list()
top_category_names = format_shop_category_name(X_train)["shop_category_name"].value_counts().head(21).index.to_list()

In [22]:
def preprocess(df: pd.DataFrame):
    df = df.drop("vendor_code", axis=1)
    df = process_description(df)
    df = process_vendor_name(df, top_vendor_names)
    df = process_shop_category_name(df, top_category_names)
    return df

In [23]:
X_train_pr = preprocess(X_train)
X_test_pr = preprocess(X_test)

In [25]:
char_tfidf = TfidfVectorizer(analyzer="char", max_features=10000).fit(X_train_pr["description"])
word_tfidf = TfidfVectorizer(analyzer="word", max_features=10000).fit(X_train_pr["description"])
char_count = CountVectorizer(analyzer="char", max_features=10000).fit(X_train_pr["description"])
word_count = CountVectorizer(analyzer="word", max_features=10000).fit(X_train_pr["description"])


In [ ]:
X_train_pr = extract_description(X_train_pr, {
    "char_tfidf": char_tfidf,
    "word_tfidf": word_tfidf,
    "char_count": char_count,
    "word_count": word_count,
})

X_test_pr = extract_description(X_test_pr, {
    "char_tfidf": char_tfidf,
    "word_tfidf": word_tfidf,
    "char_count": char_count,
    "word_count": word_count,
})